# 1.1 Pregatirea mediului

### Descarcarea tuturor librariilor

In [ ]:
%pip install -r ../requirements.txt

In [4]:
# Import necessary libraries

import re
import os
import gc
import sys
import mne
import json
import time
import dask
import zipfile
import logging
import tempfile
import importlib
import numpy as np
import pandas as pd
import pyarrow as pa
import warnings as wn
import dask.dataframe as dd
import pyarrow.parquet as pq

from pathlib import Path
from numpy.lib import format as npfmt
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple

In [5]:
# Ignore all warnings
wn.filterwarnings("ignore")

In [6]:
# Check all the packages versions

package_versions = {
    "python": sys.version.split()[0],
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "dask": dask.__version__,
    "pyarrow": pa.__version__,
    "mne": mne.__version__,
}

print(package_versions)

{'python': '3.12.3', 'numpy': '2.5.4', 'pandas': '3.1.1', 'dask': '2026.8.0', 'pyarrow': '26.0.1', 'mne': '1.13.2'}


In [7]:
# Import configuration module

if str(Path.cwd().parent) not in sys.path:
    sys.path.insert(0, str(Path.cwd().parent))

import config as cfg
importlib.reload(cfg)

<module 'config' from '/home/probook/Documents/GitHub/Mind-Map/project/config.py'>

In [8]:
# Define project root folder

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'config.py').exists() and (PROJECT_ROOT.parent / 'config.py').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

cfg.ensure_directories()

In [9]:
# Create general variables

VALIDATE_NPZ_VALUES: bool = True
CSV_CHUNK_SIZE: int = 50_000
CSV_COUNT_ALL_X_ROWS: bool = False
RUN_MOABB_REEXTRACTION: bool = False
MOABB_OVERWRITE: bool = False

rng = np.random.default_rng(cfg.RANDOM_SEED)
run_start_ts = time.perf_counter()
ingested_at_utc = datetime.now(timezone.utc).isoformat()

_limit_subjects = cfg.DEFAULT_LIMIT_SUBJECTS
_data_dir = cfg.DATA_DIR

In [11]:
# Create a logger service

log_file_path = cfg.LOGS_DIR / "00-data-acquisition.log"
logger = logging.getLogger("data-acquisition")
logger.setLevel(logging.INFO)
logger.handlers.clear()

formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
for handler in (logging.StreamHandler(sys.stdout), logging.FileHandler(log_file_path, mode="w", encoding="utf-8")):
    handler.setFormatter(formatter)
    logger.addHandler(handler)

logger.info("Project root: %s | Schema: %s | Seed: %d", cfg.to_relative_path(cfg.PROJECT_DIR), cfg.INGESTION_SCHEMA_VERSION, cfg.RANDOM_SEED)

2026-11-19 16:30:21,509 | INFO | Project root: . | Schema: 1.0.0 | Seed: 42


# 1.2 Achizitia datelor

Extragerea din MOABB (`Yang2025`, `LeftRightImagery`) este protejată prin `RUN_MOABB_REEXTRACTION = False`. Când este activată, procesează **un singur subiect per iterație**, salvează atomic NPZ + Parquet (inclusiv `session`) și eliberează memoria.

In [12]:
# Save NPZ dataset files locally 

def save_npz(target_path: Path, **arrays) -> Path:
    target = Path(target_path)
    target.parent.mkdir(parent=True, exist_ok=True)
    fd, tmp_name = tempfile.mkstemp(prefix=f".{target.name}.", suffix=".tmp", dir=str(target.parent()))
    os.close(fd)

    try:
        with open(tmp_name, "wb") as f:
            np.save_compressed(f, **arrays)
        os.replace(tmp_name, target)
    except Exception:
        if os.path.exists(tmp_name):
            os.unlink(tmp_name)
        raise
    return target

In [13]:
# Extract and save one subject from MOABB

def extract_subject_from_moabb(subject_id: int, dataset, paradigm, overwrite: bool = False) -> Dict[str, Any]:
    npz_target = cfg.get_subject_npz_path(subject_id)
    pq_target = cfg.get_subject_parquet_path(subject_id)
    
    if not overwrite and npz_target.exists() and pq_target.exists():
        return {"subject_id": subject_id, "status": "skipped_existing"}

    X, y, metadata = paradigm.get_data(dataset=dataset, subjects=[subject_id])
    n_trials, n_channels, n_timestamps = X.shape

    X_flat = np.transpose(X, (0, 2, 1)).reshape(n_trials * n_timestamps, n_channels).astype(np.float32)
    df_sub = pd.DataFrame(X_flat, columns=[f"ch_{i}" for i in range(n_channels)])
    df_sub.insert(0, "timestamp", np.tile(np.arange(n_timestamps, dtype=np.int32), n_trials))
    df_sub.insert(0, "trial", np.repeat(np.arange(n_trials, dtype=np.int32), n_timestamps))
    
    if "session" in metadata.columns:
        df_sub.insert(0, "session", np.repeat(metadata["session"].astype(str).to_numpy(), n_timestamps))
    df_sub.insert(0, "subject", np.int32(subject_id))
    df_sub.insert(0, "label", np.repeat(np.asarray(y, dtype=str), n_timestamps))

    cfg.atomic_write_parquet(df_sub, pq_target)
    atomic_save_npz(npz_target, X=X, y=y, metadata=metadata)
    
    del X, y, metadata, X_flat, df_sub
    
    gc.collect()
    return {"subject_id": subject_id, "status": "extracted"}

In [14]:
# Run MOABB extraction per subject (only when enabled)

if RUN_MOABB_REEXTRACTION:
    from moabb.datasets import Yang2025
    from moabb.paradigms import LeftRightImagery

    dataset = Yang2025()
    dataset.return_all_modalities = cfg.ALL_MOBILITIES
    paradigm = LeftRightImagery(fmin=cfg.FMIN, fmax=cfg.FMAX)
    subjects = dataset.subject_list[:_limit_subjects]

    extraction_results = [
        extract_subject_from_moabb(sub, dataset, paradigm, overwrite=MOABB_OVERWRITE)
        for sub in subjects
    ]
    print(extraction_results)
else:
    print("RUN_MOABB_REEXTRACTION=False — folosim fisierele locale existente.")

RUN_MOABB_REEXTRACTION=False — folosim fisierele locale existente.


# 1.3 Inventarierea fisierelor

Înregistrăm calea relativă, formatul, dimensiunea și data modificării pentru fiecare artefact. Dimensiunea fișierului este doar informativă și nu validează conținutul.

In [16]:
# Helpers for file discovery and metadata

SUBJECT_RE = re.compile(r"data_sub_(\d+)")


def extract_subject_id(path: Path) -> Optional[int]:
    m = SUBJECT_RE.search(path.stem)
    return int(m.group(1)) if m else None


def describe_file(path: Path, role: str, file_format: str) -> Dict[str, Any]:
    st = path.stat()
    return {
        "subject_id": extract_subject_id(path),
        "role": role,
        "format": file_format,
        "file_name": path.name,
        "source_path": cfg.to_relative_path(path),
        "file_size_bytes": int(st.st_size),
        "file_size_mb": round(st.st_size / (1024 ** 2), 3),
        "modified_at": datetime.fromtimestamp(st.st_mtime, tz=timezone.utc).isoformat(),
    }

In [25]:
# Collect NPZ, Parquet files

npz_files = cfg.get_all_npz_files()
parquet_files = cfg.get_all_parquet_files()

inventory_records: List[Dict[str, Any]] = (
    [describe_file(p, "subject_array", "npz") for p in npz_files]
    + [describe_file(p, "subject_tabular", "parquet") for p in parquet_files]
)

inventory_df = pd.DataFrame(inventory_records)

In [26]:
inventory_df

,subject_id,role,format,file_name,source_path,file_size_bytes,file_size_mb,modified_at
0,1,subject_array,npz,data_sub_01.npz,data/data_sub_01.npz,1133414750,1080.909,2026-11-14T03:33:21.526579+00:00
1,2,subject_array,npz,data_sub_02.npz,data/data_sub_02.npz,1147618561,1094.454,2026-11-14T03:39:08.205976+00:00
2,3,subject_array,npz,data_sub_03.npz,data/data_sub_03.npz,1127465237,1075.235,2026-11-14T03:44:59.816091+00:00
3,1,subject_tabular,parquet,data_sub_01.parquet,data/data_sub_01.parquet,705006239,672.346,2026-11-14T03:32:37.879519+00:00
4,2,subject_tabular,parquet,data_sub_02.parquet,data/data_sub_02.parquet,708609835,675.783,2026-11-14T03:38:24.028908+00:00
5,3,subject_tabular,parquet,data_sub_03.parquet,data/data_sub_03.parquet,700647865,668.190,2026-11-14T03:44:16.253550+00:00


In [27]:
# Check missing pairs or duplicate subject IDs

npz_subjects = [extract_subject_id(p) for p in npz_files if extract_subject_id(p) is not None]
parquet_subjects = [extract_subject_id(p) for p in parquet_files if extract_subject_id(p) is not None]
all_subjects = sorted(set(npz_subjects) | set(parquet_subjects))

missing_npz = sorted(set(parquet_subjects) - set(npz_subjects))
missing_parquet = sorted(set(npz_subjects) - set(parquet_subjects))
duplicate_subjects = (len(npz_subjects) != len(set(npz_subjects))) or (len(parquet_subjects) != len(set(parquet_subjects)))


In [28]:
# Save to the logger service

logger.info("Subiecti: %s | Lipsa NPZ: %s | Lipsa Parquet: %s", all_subjects, missing_npz, missing_parquet)
{"all_subjects": all_subjects, "missing_npz": missing_npz, "missing_parquet": missing_parquet, "duplicate_subjects": duplicate_subjects}

2026-11-19 16:40:33,644 | INFO | Subiecti: [1, 2, 3] | Lipsa NPZ: [] | Lipsa Parquet: []


{'all_subjects': [1, 2, 3],
 'missing_npz': [],
 'missing_parquet': [],
 'duplicate_subjects': False}

# 1.4 Inspectarea fisierelor NPZ (secvential)

Inspectăm formele din headerul `.npy` fără decompresie, apoi încărcăm pe rând datele **unui singur subiect** și eliberăm memoria înainte de următorul.

In [31]:
# Read NPZ array headers without uncompressing

def read_npz_headers(npz_path: Path) -> Dict[str, Dict[str, Any]]:
    headers: Dict[str, Dict[str, Any]] = {}
    
    with zipfile.ZipFile(npz_path, "r") as zf:
        for member in zf.namelist():
            
            if not member.endswith(".npy"):
                continue
            with zf.open(member) as f:
                ver = npfmt.read_magic(f)
                reader = npfmt.read_array_header_1_0 if ver[0] == 1 else npfmt.read_array_header_2_0
                shape, _, dtype = reader(f)
            headers[member[:-4]] = {"shape": tuple(int(v) for v in shape), "dtype": str(dtype)}
    
    return headers

In [33]:
# Infer sampling frequency from README trial interval

def sfreq_from_samples(n_samples: int) -> Tuple[Optional[float], str]:
    duration_s = cfg.EXPECTED_TRIAL_INTERVAL[1] - cfg.EXPECTED_TRIAL_INTERVAL[0]
    expected_samples = int(duration_s * cfg.EXPECTED_SFREQ_HZ)
    
    if n_samples in (expected_samples, expected_samples + 1):
        return float(cfg.EXPECTED_SFREQ_HZ), "inferred_from_readme_interval_and_n_samples"
    
    return None, "unknown"

In [34]:
# Read each npz file format 

def _read_npz_contents(
    npz_path: Path, 
    subject_id: Optional[int], 
    validate_values: bool = True
) -> Tuple[Dict[str, int], List[str], List[str], Optional[bool], 
    Optional[float], Optional[float], List[str]]:

    label_counts: Dict[str, int] = {}
    sessions: List[str] = []
    runs: List[str] = []
    x_all_finite: Optional[bool] = None
    x_min: Optional[float] = None
    x_max: Optional[float] = None
    discrepancies: List[str] = []

    with np.load(npz_path, allow_pickle=True) as z:
        
        if "y" in z:
            uniq_y, cnt_y = np.unique(z["y"], return_counts=True)
            label_counts = {str(k): int(v) for k, v in zip(uniq_y, cnt_y)}

        if "metadata" in z:
            meta = z["metadata"]

            if meta.ndim == 2 and meta.shape[1] >= 3:
                sessions = sorted({str(v) for v in meta[:, 1]})
                runs = sorted({str(v) for v in meta[:, 2]})
                meta_subjs = sorted({int(v) for v in meta[:, 0]})

                if subject_id is not None and meta_subjs != [subject_id]:
                    discrepancies.append(f"Subject ID metadata {meta_subjs} != {subject_id}")

        if validate_values and "X" in z:
            X = z["X"]
            x_all_finite = bool(np.isfinite(X).all())
            x_min, x_max = float(np.min(X)), float(np.max(X))

            if not x_all_finite:
                discrepancies.append("Valori NaN/Inf in X")
            del X

    gc.collect()
    return label_counts, sessions, runs, x_all_finite, x_min, x_max, discrepancies

In [36]:
# Inspeact each npz file content

def inspect_npz(npz_path: Path, validate_values: bool = True) -> Dict[str, Any]:
    subject_id = extract_subject_id(npz_path)
    headers = read_npz_headers(npz_path)
    discrepancies: List[str] = []

    missing_keys = sorted({"X", "y", "metadata"} - set(headers.keys()))
    if missing_keys:
        discrepancies.append(f"Chei NPZ lipsa: {missing_keys}")

    x_shape = headers.get("X", {}).get("shape", ())
    n_trials = int(x_shape[0]) if len(x_shape) >= 1 else 0
    n_channels = int(x_shape[1]) if len(x_shape) >= 2 else 0
    n_samples = int(x_shape[2]) if len(x_shape) >= 3 else 0

    (
        label_counts,
        sessions,
        runs,
        x_all_finite,
        x_min,
        x_max,
        content_discrepancies,
    ) = _read_npz_contents(npz_path, subject_id, validate_values=validate_values)
    discrepancies.extend(content_discrepancies)

    sfreq, sfreq_source = sfreq_from_samples(n_samples)
    if sfreq is None:
        discrepancies.append(f"n_samples={n_samples} incompatibil cu README")

    if n_channels != cfg.NUM_CHANNELS:
        discrepancies.append(f"n_channels={n_channels} != {cfg.NUM_CHANNELS}")

    schema_ok = (
        (not missing_keys)
        and n_trials > 0
        and n_channels == cfg.NUM_CHANNELS
        and (x_all_finite is not False)
    )

    return {
        "subject_id": subject_id,
        "source_path": cfg.to_relative_path(npz_path),
        "format": "npz",
        "x_shape": str(x_shape),
        "x_dtype": headers.get("X", {}).get("dtype"),
        "n_trials": n_trials,
        "n_channels": n_channels,
        "n_samples": n_samples,
        "sfreq": sfreq,
        "sfreq_verified": False,
        "sfreq_source": sfreq_source,
        "labels": sorted(label_counts.keys()),
        "label_counts": label_counts,
        "sessions": sessions,
        "runs": runs,
        "has_session_metadata": len(sessions) > 0,
        "x_all_finite": x_all_finite,
        "x_min": x_min,
        "x_max": x_max,
        "schema_ok": schema_ok,
        "discrepancies": discrepancies,
    }

In [37]:
# Run sequential NPZ inspection across discovered subjects

npz_audit_results: List[Dict[str, Any]] = []
for p in npz_files:
    t0 = time.perf_counter()
    res = inspect_npz(p, validate_values=VALIDATE_NPZ_VALUES)
    npz_audit_results.append(res)
    logger.info("NPZ sub_%02d (%.2fs) | shape=%s | sessions=%s | finite=%s", res["subject_id"], time.perf_counter() - t0, res["x_shape"], res["sessions"], res["x_all_finite"])

2026-11-19 17:06:43,903 | INFO | NPZ sub_01 (13.86s) | shape=(600, 64, 4001) | sessions=['0', '1', '2'] | finite=True
2026-11-19 17:06:59,535 | INFO | NPZ sub_02 (15.63s) | shape=(600, 64, 4001) | sessions=['0', '1', '2'] | finite=True
2026-11-19 17:07:13,601 | INFO | NPZ sub_03 (14.06s) | shape=(600, 64, 4001) | sessions=['0', '1', '2'] | finite=True


In [38]:
# Display the npz audit dataframe

npz_audit_df = pd.DataFrame(npz_audit_results)
npz_audit_df[["subject_id", "x_shape", "x_dtype", "n_trials", "sfreq", "sessions", "x_all_finite", "schema_ok"]]

,subject_id,x_shape,x_dtype,n_trials,sfreq,sessions,x_all_finite,schema_ok
0,1,"(600, 64, 4001)",float64,600,1000.0,"[0, 1, 2]",True,True
1,2,"(600, 64, 4001)",float64,600,1000.0,"[0, 1, 2]",True,True
2,3,"(600, 64, 4001)",float64,600,1000.0,"[0, 1, 2]",True,True


# 1.5 Audit Parquet cu Dask

Citire leneșă cu `dask.dataframe.read_parquet` și verificări de schemă, partiții și valori null într-o singură trecere per subiect.

In [39]:
# Load all Parquet files lazily with Dask

print(f"Files: {[f.name for f in parquet_files]}")
data = dd.read_parquet(parquet_files, engine=cfg.PARQUET_ENGINE)
dask_overview = {
    "npartitions_total": int(data.npartitions),
    "n_columns": int(len(data.columns)),
    "dtypes_summary": data.dtypes.astype(str).value_counts().to_dict(),
}

print(dask_overview)

Files: ['data_sub_01.parquet', 'data_sub_02.parquet', 'data_sub_03.parquet']
{'npartitions_total': 9, 'n_columns': 68, 'dtypes_summary': {'float32': 64, 'int32': 3, 'string': 1}}


In [42]:
# Preview first 10 rows with mapped channel names
data.rename(columns=cfg.CHANNEL_MAP).head(n=10)

,label,subject,trial,timestamp,Fpz,Fp1,Fp2,AF3,AF4,AF7,...,PO7,PO8,Oz,O1,O2,ECG,EOG1,EOG2,EOG3,EOG4
0,left_hand,1,0,0,-0.002103,-0.001852,-0.002874,-0.003164,-0.001652,-0.001154,...,0.001198,0.002832,0.003152,0.002958,0.002928,-375.0,-21.107891,-115.354019,-74.953644,-375.0
1,left_hand,1,0,1,-0.002297,-0.002012,-0.002857,-0.003467,-0.001647,-0.001074,...,0.001058,0.002794,0.003080,0.002912,0.002866,-375.0,-21.108248,-115.354019,-72.998672,-375.0
2,left_hand,1,0,2,-0.002465,-0.002167,-0.002790,-0.003735,-0.001618,-0.000980,...,0.000904,0.002692,0.002942,0.002804,0.002741,-375.0,-21.107445,-115.354370,-71.007172,-375.0
3,left_hand,1,0,3,-0.002604,-0.002315,-0.002677,-0.003964,-0.001563,-0.000877,...,0.000742,0.002534,0.002746,0.002643,0.002563,-375.0,-21.108473,-115.353523,-69.126808,-375.0
4,left_hand,1,0,4,-0.002713,-0.002453,-0.002523,-0.004151,-0.001481,-0.000769,...,0.000578,0.002331,0.002504,0.002439,0.002342,-375.0,-21.108650,-115.353256,-67.201118,-375.0
5,left_hand,1,0,5,-0.002791,-0.002581,-0.002331,-0.004293,-0.001374,-0.000662,...,0.000418,0.002094,0.002227,0.002203,0.002088,-375.0,-21.108070,-115.352676,-65.306587,-375.0
6,left_hand,1,0,6,-0.002836,-0.002694,-0.002108,-0.004388,-0.001242,-0.000559,...,0.000266,0.001832,0.001928,0.001945,0.001815,-375.0,-21.107578,-115.352051,-64.069191,-375.0
7,left_hand,1,0,7,-0.002850,-0.002792,-0.001860,-0.004436,-0.001085,-0.000465,...,0.000128,0.001557,0.001618,0.001677,0.001532,-375.0,-21.106148,-115.353882,-63.527878,-375.0
8,left_hand,1,0,8,-0.002832,-0.002872,-0.001593,-0.004435,-0.000906,-0.000383,...,0.000008,0.001280,0.001309,0.001411,0.001252,-375.0,-21.106237,-115.354729,-63.690777,-375.0
9,left_hand,1,0,9,-0.002783,-0.002932,-0.001315,-0.004388,-0.000708,-0.000318,...,-0.000090,0.001012,0.001012,0.001155,0.000984,-375.0,-21.107489,-115.355133,-64.224182,-375.0


In [44]:
# Read each file and get some results

EXPECTED_PARQUET_COLS = ["label", "subject", "trial", "timestamp"] + cfg.RAW_CHANNEL_NAMES


def _compute_parquet_stats(
    ddf: dd.DataFrame,
) -> Tuple[int, int, int, List[int], Dict[str, int]]:

    null_total, n_trials, ts_min, ts_max, subj_vals, label_row_counts = dask.compute(
        ddf.isnull().sum().sum(),
        ddf["trial"].nunique(),
        ddf["timestamp"].min(),
        ddf["timestamp"].max(),
        ddf["subject"].unique(),
        ddf.groupby("label").size(),
    )

    n_trials = int(n_trials)
    n_samples = int(ts_max) - int(ts_min) + 1 if n_trials > 0 else 0
    null_total = int(null_total)
    subj_list = sorted(int(v) for v in subj_vals.tolist())
    
    label_counts = {
        str(lbl): int(cnt // n_samples) if n_samples > 0 else int(cnt)
        for lbl, cnt in label_row_counts.to_dict().items()
    }

    return null_total, n_trials, n_samples, subj_list, label_counts

In [47]:
# audit the content of each file

def audit_one_parquet(pq_path: Path) -> Dict[str, Any]:

    subject_id = extract_subject_id(pq_path)
    pf = pq.ParquetFile(pq_path)
    n_rows = int(pf.metadata.num_rows)
    num_row_groups = int(pf.metadata.num_row_groups)

    ddf = dd.read_parquet(pq_path, engine=cfg.PARQUET_ENGINE)
    cols = list(ddf.columns)
    discrepancies: List[str] = []
    has_session_col = ("session" in cols)
    
    if not has_session_col:
        discrepancies.append("Coloana 'session' lipseste din Parquet (disponibila in NPZ metadata)")
    if cols != EXPECTED_PARQUET_COLS:
        discrepancies.append("Schema coloanelor Parquet difera de cea asteptata")

    null_total, n_trials, n_samples, subj_list, label_counts = _compute_parquet_stats(ddf)

    if null_total > 0:
        discrepancies.append(f"{null_total} valori null in Parquet")
    
    if subject_id is not None and subj_list != [subject_id]:
        discrepancies.append(f"subject={subj_list} != [{subject_id}]")
    
    if n_trials * n_samples != n_rows:
        discrepancies.append(f"n_trials*n_samples ({n_trials * n_samples}) != n_rows ({n_rows})")

    schema_ok = (cols == EXPECTED_PARQUET_COLS) and (null_total == 0) and (n_trials * n_samples == n_rows)

    return {
        "subject_id": subject_id,
        "source_path": cfg.to_relative_path(pq_path),
        "format": "parquet",
        "n_rows": n_rows,
        "npartitions": int(ddf.npartitions),
        "num_row_groups": num_row_groups,
        "n_trials": n_trials,
        "n_channels": len([c for c in cols if c.startswith("ch_")]),
        "n_samples": n_samples,
        "null_total": null_total,
        "labels": sorted(label_counts.keys()),
        "label_counts": label_counts,
        "has_session_metadata": has_session_col,
        "schema_ok": schema_ok,
        "discrepancies": discrepancies,
    }

In [48]:
# Run Parquet audit per subject

parquet_audit_results: List[Dict[str, Any]] = []
for p in parquet_files:
    t0 = time.perf_counter()
    res = audit_one_parquet(p)
    parquet_audit_results.append(res)
    logger.info("Parquet sub_%02d (%.2fs) | rows=%d | trials=%d | samples=%d", 
                res["subject_id"], time.perf_counter() - t0, res["n_rows"], 
                res["n_trials"], res["n_samples"])

2026-11-19 17:18:56,391 | INFO | Parquet sub_01 (6.04s) | rows=2400600 | trials=600 | samples=4001
2026-11-19 17:19:07,674 | INFO | Parquet sub_02 (11.26s) | rows=2400600 | trials=600 | samples=4001
2026-11-19 17:19:18,381 | INFO | Parquet sub_03 (10.71s) | rows=2400600 | trials=600 | samples=4001


In [49]:
# Display the audit the parquet files
parquet_audit_df = pd.DataFrame(parquet_audit_results)
parquet_audit_df[["subject_id", "n_rows", "npartitions", "num_row_groups", "n_trials", "n_channels", "n_samples", "null_total", "schema_ok"]]

,subject_id,n_rows,npartitions,num_row_groups,n_trials,n_channels,n_samples,null_total,schema_ok
0,1,2400600,3,3,600,64,4001,0,True
1,2,2400600,3,3,600,64,4001,0,True
2,3,2400600,3,3,600,64,4001,0,True


# 1.6 Verificarea maparii canalelor (`ch_*`, EEG / ECG / EOG)

Confirmăm separarea celor 59 canale EEG față de ECG și EOG și compatibilitatea cu standardul notatiilor medicale.

In [50]:
# Validate channel mapping and summarize signal types

channel_counts = cfg.get_channel_type_counts()
channel_map_valid = (
    len(cfg.RAW_CHANNEL_NAMES) == cfg.NUM_CHANNELS == 64
    and channel_counts == {"eeg": 59, "ecg": 1, "eog": 4, "total": 64}
    and list(cfg.CHANNEL_MAP.keys()) == cfg.RAW_CHANNEL_NAMES
    and list(cfg.CHANNEL_MAP.values()) == cfg.MED_COLUMNS
)

pd.DataFrame({
    "raw_col": cfg.RAW_CHANNEL_NAMES,
    "channel_name": cfg.MED_COLUMNS,
    "signal_type": [cfg.CHANNEL_TYPES[ch] for ch in cfg.MED_COLUMNS],
}).groupby("signal_type").agg(count=("channel_name", "count"), first=("channel_name", "first"), last=("channel_name", "last"))

,count,first,last
signal_type,,,
ecg,1,ECG,ECG
eeg,59,Fpz,O2
eog,4,EOG1,EOG4


In [51]:
# Read the channel order from the readme

def parse_readme_eeg_channels(readme_path: Path) -> List[str]:
    for line in readme_path.read_text(encoding="utf-8").splitlines():
        
        if line.strip().startswith("Channel names:"):
            return [c.strip() for c in line.split(":", 1)[1].split(",") if c.strip()]
    return []

In [52]:
readme_eeg = parse_readme_eeg_channels(_data_dir / "README.md")
montage_names = set(mne.channels.make_standard_montage("standard_1005").ch_names)
eeg_missing_in_montage = [ch for ch in cfg.EEG_CHANNELS if ch not in montage_names]

channel_audit = {
    "channel_type_counts": channel_counts,
    "channel_map_valid": channel_map_valid,
    "readme_matches_config_eeg": (readme_eeg == cfg.EEG_CHANNELS),
    "eeg_missing_in_standard_1005": eeg_missing_in_montage,
    "historical_note_discrepancy": "Nota din prototip mentioneaza 53 canale, dar datele si config.py au 64 canale (59 EEG + 1 ECG + 4 EOG).",
}
logger.info("Audit canale: %s", channel_audit)

2026-11-19 18:44:34,176 | INFO | Audit canale: {'channel_type_counts': {'eeg': 59, 'ecg': 1, 'eog': 4, 'total': 64}, 'channel_map_valid': True, 'readme_matches_config_eeg': True, 'eeg_missing_in_standard_1005': [], 'historical_note_discrepancy': 'Nota din prototip mentioneaza 53 canale, dar datele si config.py au 64 canale (59 EEG + 1 ECG + 4 EOG).'}


In [53]:
channel_audit

{'channel_type_counts': {'eeg': 59, 'ecg': 1, 'eog': 4, 'total': 64},
 'channel_map_valid': True,
 'readme_matches_config_eeg': True,
 'eeg_missing_in_standard_1005': [],
 'historical_note_discrepancy': 'Nota din prototip mentioneaza 53 canale, dar datele si config.py au 64 canale (59 EEG + 1 ECG + 4 EOG).'}

# 1.7 Provenienta fisierelor  auxiliare CSV (X_data.csv, y_data.csv)

`X_data.csv` (~8,9 GB) nu se încarcă complet în pandas deci citim doar primul chunk și verificăm corespondența cu `data_sub_01.parquet` și `y_data.csv` pentru potrivire.

In [54]:
# Read only the first chunk of X_data.csv and y_data.csv

x_csv_path = _data_dir / "X_data.csv"
y_csv_path = _data_dir / "y_data.csv"
csv_provenance: Dict[str, Any] = {"x_csv_exists": x_csv_path.exists(), "y_csv_exists": y_csv_path.exists()}

if x_csv_path.exists() and y_csv_path.exists():
    
    x_chunk = next(pd.read_csv(x_csv_path, chunksize=min(CSV_CHUNK_SIZE, 1_000)))
    y_chunk = next(pd.read_csv(y_csv_path, chunksize=min(CSV_CHUNK_SIZE, 1_000)))
    x_cols = list(x_chunk.columns)
    x_ch_cols = [c for c in x_cols if c.startswith("ch_")]
    
    csv_provenance.update({
        "x_columns_count": len(x_cols),
        "x_non_channel_columns": [c for c in x_cols if not c.startswith("ch_")],
        "x_channel_count": len(x_ch_cols),
        "x_channel_range": f"{x_ch_cols[0]}..{x_ch_cols[-1]}" if x_ch_cols else None,
        "y_columns": list(y_chunk.columns),
    })

In [55]:
# SHow the provence
csv_provenance

{'x_csv_exists': True,
 'y_csv_exists': True,
 'x_columns_count': 61,
 'x_non_channel_columns': ['trial', 'timestamp'],
 'x_channel_count': 59,
 'x_channel_range': 'ch_0..ch_58',
 'y_columns': ['label']}

In [60]:
# Count function in y and X

def count_csv_data_rows(file_path: Path, buf_size: int = 8 * 1024 * 1024) -> int:
    lines, last_byte = 0, b""
    
    with open(file_path, "rb") as f:
        while chunk := f.read(buf_size):
            lines += chunk.count(b"\n")
            last_byte = chunk[-1:]
    
    if last_byte and last_byte != b"\n":
        lines += 1
    return max(0, lines - 1)

In [61]:
# Count y_data.csv rows and X_data.csv in binary chunks

if y_csv_path.exists():
    csv_provenance["y_data_rows"] = count_csv_data_rows(y_csv_path)
    csv_provenance["parquet_rows_total"] = int(parquet_audit_df["n_rows"].sum())
    csv_provenance["y_matches_parquet_total"] = (csv_provenance["y_data_rows"] == csv_provenance["parquet_rows_total"])

csv_provenance["x_data_rows"] = count_csv_data_rows(x_csv_path) if (x_csv_path.exists() and CSV_COUNT_ALL_X_ROWS) else "skipped (CSV_COUNT_ALL_X_ROWS=False)"
csv_provenance["conclusion"] = (
    "X_data.csv si y_data.csv provin din exportul concatenat al celor 3 subiecti (7.201.800 randuri), "
    "insa X_data.csv contine doar cele 59 canale EEG (ch_0..ch_58), fara ECG/EOG, fara subject si fara session."
)

logger.info("Provenienta CSV: %s", csv_provenance)

2026-11-19 18:52:22,113 | INFO | Provenienta CSV: {'x_csv_exists': True, 'y_csv_exists': True, 'x_columns_count': 61, 'x_non_channel_columns': ['trial', 'timestamp'], 'x_channel_count': 59, 'x_channel_range': 'ch_0..ch_58', 'y_columns': ['label'], 'sub01_head_max_abs_diff': 1.988336318509143e-10, 'sub01_head_labels_match': True, 'y_data_rows': 7201800, 'parquet_rows_total': 7201800, 'y_matches_parquet_total': True, 'x_data_rows': 'skipped (CSV_COUNT_ALL_X_ROWS=False)', 'conclusion': 'X_data.csv si y_data.csv provin din exportul concatenat al celor 3 subiecti (7.201.800 randuri), insa X_data.csv contine doar cele 59 canale EEG (ch_0..ch_58), fara ECG/EOG, fara subject si fara session.'}


In [64]:
csv_provenance

{'x_csv_exists': True,
 'y_csv_exists': True,
 'x_columns_count': 61,
 'x_non_channel_columns': ['trial', 'timestamp'],
 'x_channel_count': 59,
 'x_channel_range': 'ch_0..ch_58',
 'y_columns': ['label'],
 'sub01_head_max_abs_diff': 1.988336318509143e-10,
 'sub01_head_labels_match': True,
 'y_data_rows': 7201800,
 'parquet_rows_total': 7201800,
 'y_matches_parquet_total': True,
 'x_data_rows': 'skipped (CSV_COUNT_ALL_X_ROWS=False)',
 'conclusion': 'X_data.csv si y_data.csv provin din exportul concatenat al celor 3 subiecti (7.201.800 randuri), insa X_data.csv contine doar cele 59 canale EEG (ch_0..ch_58), fara ECG/EOG, fara subject si fara session.'}

# 1.8 Generarea manifestului si rapoartelor

Construim `data/manifest.parquet` pentru informatii suplimentare și sumarele din `reports/` prin scriere atomică. Un subiect este `usable = True` doar dacă NPZ și Parquet sunt lizibile, valide și concordante.

In [66]:
# Check consistency between NPZ and Parquet for one subject

def check_pair_consistency(npz_res: Optional[Dict[str, Any]], pq_res: Optional[Dict[str, Any]]) -> List[str]:
    issues: List[str] = []
    if npz_res is None:
        issues.append("Lipseste NPZ")
    if pq_res is None:
        issues.append("Lipseste Parquet")
    if npz_res is not None and pq_res is not None:
        for field in ("n_trials", "n_channels", "n_samples", "label_counts"):
            if npz_res[field] != pq_res[field]:
                issues.append(f"{field} diferit: NPZ={npz_res[field]} vs Parquet={pq_res[field]}")
    return issues

In [67]:
# Build function for manifest dataframe 

def build_manifest_df() -> pd.DataFrame:
    inv_lookup = {}
    for record in inventory_records:
        if record["subject_id"] is not None:
            inv_lookup[(record["subject_id"], record["format"])] = record

    npz_lookup = {r["subject_id"]: r for r in npz_audit_results}
    pq_lookup = {r["subject_id"]: r for r in parquet_audit_results}

    ch_types_json = json.dumps(channel_counts, ensure_ascii=False)
    n_eeg = len(cfg.EEG_CHANNELS)
    n_aux = len(cfg.ECG_CHANNELS) + len(cfg.EOG_CHANNELS)
    rows = []

    for sub_id in all_subjects:
        npz_res = npz_lookup.get(sub_id)
        pq_res = pq_lookup.get(sub_id)

        pair_issues = check_pair_consistency(npz_res, pq_res)
        pair_ok = (npz_res is not None) and (pq_res is not None) and (not pair_issues)

        npz_data = npz_res or {}
        sessions = ",".join(npz_data.get("sessions", []))
        runs = ",".join(npz_data.get("runs", []))
        sfreq = float(npz_data["sfreq"]) if npz_data.get("sfreq") is not None else np.nan

        for fmt_name, res in [("npz", npz_res), ("parquet", pq_res)]:
            if res is None:
                continue

            inv = inv_lookup[(sub_id, fmt_name)]
            sfreq_source = npz_data.get("sfreq_source", "unknown") if fmt_name == "npz" else "from_npz_pair"

            rows.append({
                "subject_id": int(sub_id),
                "source_path": inv["source_path"],
                "format": fmt_name,
                "file_size_bytes": inv["file_size_bytes"],
                "file_size_mb": inv["file_size_mb"],
                "modified_at": inv["modified_at"],
                "n_trials": res["n_trials"],
                "n_channels": res["n_channels"],
                "n_eeg_channels": n_eeg,
                "n_aux_channels": n_aux,
                "n_samples": res["n_samples"],
                "sfreq": sfreq,
                "sfreq_verified": res.get("sfreq_verified", False),
                "sfreq_source": sfreq_source,
                "labels": ",".join(res["labels"]),
                "label_counts_json": json.dumps(res["label_counts"], ensure_ascii=False, sort_keys=True),
                "sessions": sessions,
                "runs": runs,
                "has_session_metadata": res["has_session_metadata"],
                "channel_types_json": ch_types_json,
                "channel_order_ok": bool(channel_map_valid),
                "schema_ok": res["schema_ok"],
                "usable": bool(res["schema_ok"] and pair_ok and channel_map_valid),
                "discrepancies": " | ".join(res["discrepancies"] + pair_issues),
                "ingestion_version": cfg.INGESTION_SCHEMA_VERSION,
                "ingested_at": ingested_at_utc,
            })

    df = pd.DataFrame(rows)
    return df.sort_values(["subject_id", "format"]).reset_index(drop=True)

In [68]:
# Call the manifest function

manifest_df = build_manifest_df()
manifest_df[["subject_id", "format", "n_trials", "n_channels", "n_samples", "sfreq", "sessions", "usable", "discrepancies"]]

,subject_id,format,n_trials,n_channels,n_samples,sfreq,sessions,usable,discrepancies
0,1,npz,600,64,4001,1000.0,"0,1,2",True,
1,1,parquet,600,64,4001,1000.0,"0,1,2",True,Coloana 'session' lipseste din Parquet (dispon...
2,2,npz,600,64,4001,1000.0,"0,1,2",True,
3,2,parquet,600,64,4001,1000.0,"0,1,2",True,Coloana 'session' lipseste din Parquet (dispon...
4,3,npz,600,64,4001,1000.0,"0,1,2",True,
5,3,parquet,600,64,4001,1000.0,"0,1,2",True,Coloana 'session' lipseste din Parquet (dispon...


In [70]:
# Save manifest and manifest_summary atomically

cfg.atomic_write_parquet(manifest_df, cfg.MANIFEST_PATH)
summary_csv_path = cfg.METRICS_DIR / "manifest_summary.csv"
cfg.atomic_write_text(summary_csv_path, manifest_df.to_csv(index=False))

logger.info("Manifest salvat la %s (%d randuri)", cfg.to_relative_path(cfg.MANIFEST_PATH), len(manifest_df))

2026-11-19 18:58:51,107 | INFO | Manifest salvat la data/manifest.parquet (6 randuri)


In [75]:
# Save execution log JSON

run_elapsed_s = round(time.perf_counter() - run_start_ts, 3)
usable_subjects = sorted(manifest_df.loc[manifest_df["usable"], "subject_id"].unique().tolist())

log_payload = {
    "notebook": "00-data-acquisition.ipynb",
    "ingested_at_utc": ingested_at_utc,
    "ingestion_version": cfg.INGESTION_SCHEMA_VERSION,
    "random_seed": cfg.RANDOM_SEED,
    "elapsed_seconds": run_elapsed_s,
    "parameters": {
        "VALIDATE_NPZ_VALUES": VALIDATE_NPZ_VALUES,
        "CSV_CHUNK_SIZE": CSV_CHUNK_SIZE,
        "CSV_COUNT_ALL_X_ROWS": CSV_COUNT_ALL_X_ROWS,
        "RUN_MOABB_REEXTRACTION": RUN_MOABB_REEXTRACTION,
    },
    
    "package_versions": package_versions,
    "subjects_discovered": all_subjects,
    "subjects_usable": usable_subjects,
    "dask_overview": dask_overview,
    "channel_audit": channel_audit,
    "csv_provenance": csv_provenance,
    "outputs": {
        "manifest_parquet": cfg.to_relative_path(cfg.MANIFEST_PATH),
        "summary_csv": cfg.to_relative_path(summary_csv_path),
    },
}

In [76]:
cfg.atomic_write_text(cfg.LOGS_DIR / "00-data-acquisition.json", json.dumps(log_payload, indent=2, ensure_ascii=False))
logger.info("Notebook finalizat in %.2fs", run_elapsed_s)

{"elapsed_seconds": run_elapsed_s, "subjects_usable": usable_subjects}

2026-11-19 19:00:31,692 | INFO | Notebook finalizat in 9085.04s


{'elapsed_seconds': 9085.036, 'subjects_usable': [1, 2, 3]}